# Notebook 02 - Pooled 5-stock LSTM smoke test

This notebook is an orchestration smoke test for the current `ml_utils` public API. It loads the five processed stock CSVs, prepares per-ticker datasets, fits one train-only global scaler, pools windowed datasets after per-ticker window construction, trains an LSTM smoke model, compares public baseline metrics, and runs a shuffled-label sanity check.

If this notebook reveals a public API blocker, stop and return to the matching `ml_utils` module for a tested fix.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "ml_utils").is_dir():
    for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
        if (candidate / "ml_utils").is_dir() and (candidate / "notebooks").is_dir():
            PROJECT_ROOT = candidate
            break
    else:
        raise RuntimeError(f"Could not locate project root from cwd={Path.cwd()}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from torch.utils.data import ConcatDataset, DataLoader, TensorDataset

from ml_utils.config import DataConfig, ModelConfig, TrainConfig, WindowConfig
from ml_utils.dataset import WindowedClassificationDataset
from ml_utils.dataset import fit_scaler_on_train
from ml_utils.dataset import make_binary_labels_from_future_avg_return
from ml_utils.dataset import make_time_splits
from ml_utils.dataset import transform_split
from ml_utils.dataset import trim_labels_at_split_boundary
from ml_utils.metrics import always_predict_baseline_metrics
from ml_utils.metrics import dummy_baseline_metrics
from ml_utils.models.lstm_classifier import LSTMClassifier
from ml_utils.seed import seed_everything
from ml_utils.trainer import Trainer, evaluate

In [ ]:
FULL_RUN = False
MAX_ROWS_PER_TICKER = 5000
NUM_EPOCHS = 2
SEED = 42
TICKERS = ["CSCO", "JPM", "KO", "MSFT", "WMT"]
DATA_DIR = PROJECT_ROOT / "data"
FEATURE_COLS = ["open", "high", "low", "close", "volume"]
TICKER_COL = "ticker"
LABEL_COL = "label"
SCALER_TYPE = "standard"
NUM_WORKERS = 0
CHECKPOINT_ROOT = PROJECT_ROOT / "checkpoints" / "notebook_02_pooled_lstm_smoke"

seed_everything(SEED, deterministic=True)

data_config = DataConfig(
    tickers=TICKERS,
    data_dir=str(DATA_DIR),
    timestamp_col="timestamp",
    price_col="close",
    feature_cols=FEATURE_COLS,
    train_ratio=0.7,
    val_ratio=0.15,
    timezone_policy="naive",
)
window_config = WindowConfig(window_size=24, label_horizon_k=12, stride=1)
train_config = TrainConfig(
    batch_size=256,
    num_epochs=NUM_EPOCHS,
    learning_rate=1e-3,
    weight_decay=0.0,
    grad_clip=None,
    early_stop_patience=NUM_EPOCHS,
    monitor_metric="val_macro_f1",
    monitor_mode="max",
    device="cpu",
    seed=SEED,
)
model_config = ModelConfig(
    name="lstm_classifier",
    params={
        "input_size": len(FEATURE_COLS),
        "hidden_size": 16,
        "num_layers": 1,
        "num_classes": 2,
        "dropout": 0.0,
        "bidirectional": False,
    },
)

print(
    "pooled smoke config | "
    f"seed={SEED} | tickers={','.join(TICKERS)} | full_run={FULL_RUN} | "
    f"rows_cap={MAX_ROWS_PER_TICKER} | epochs={train_config.num_epochs} | "
    f"batch_size={train_config.batch_size} | device={train_config.device}"
)

In [ ]:
csv_paths = {ticker: DATA_DIR / f"{ticker}.csv" for ticker in TICKERS}
missing_paths = [path for path in csv_paths.values() if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing processed CSV files: {missing_paths}")

path_table = pd.DataFrame(
    [{"ticker": ticker, "path": str(path.relative_to(PROJECT_ROOT))} for ticker, path in csv_paths.items()]
)
path_table

In [ ]:
required_cols = [data_config.timestamp_col, *data_config.feature_cols]
raw_frames = {}
load_rows = []

for ticker, path in csv_paths.items():
    frame = pd.read_csv(path)
    missing_cols = [column for column in required_cols if column not in frame.columns]
    if missing_cols:
        raise ValueError(f"{ticker} missing required columns: {missing_cols}")

    frame = frame[required_cols].copy()
    frame[data_config.timestamp_col] = pd.to_datetime(frame[data_config.timestamp_col], errors="raise")
    frame = frame.sort_values(data_config.timestamp_col).reset_index(drop=True)
    if not FULL_RUN:
        frame = frame.head(MAX_ROWS_PER_TICKER).copy()
    frame[TICKER_COL] = ticker
    raw_frames[ticker] = frame
    load_rows.append(
        {
            "ticker": ticker,
            "rows": len(frame),
            "timestamp_min": frame[data_config.timestamp_col].min(),
            "timestamp_max": frame[data_config.timestamp_col].max(),
        }
    )

load_table = pd.DataFrame(load_rows)
load_table

In [ ]:
labeled_frames = {}
label_rows = []

for ticker, frame in raw_frames.items():
    labeled = make_binary_labels_from_future_avg_return(
        frame,
        price_col=data_config.price_col,
        k=window_config.label_horizon_k,
    )
    labeled_frames[ticker] = labeled
    label_rows.append(
        {
            "ticker": ticker,
            "rows": len(labeled),
            "label_nan": int(labeled[LABEL_COL].isna().sum()),
            "label_up_rate": float(labeled[LABEL_COL].dropna().mean()),
        }
    )

label_table = pd.DataFrame(label_rows)
label_table

In [ ]:
split_frames = {}
split_rows = []

for ticker, labeled in labeled_frames.items():
    train_df, val_df, test_df = make_time_splits(
        labeled,
        train_ratio=data_config.train_ratio,
        val_ratio=data_config.val_ratio,
        timestamp_col=data_config.timestamp_col,
        timezone_policy=data_config.timezone_policy,
    )
    split_frames[ticker] = {"train": train_df, "val": val_df, "test": test_df}
    for split_name, split_df in split_frames[ticker].items():
        split_rows.append(
            {
                "ticker": ticker,
                "split": split_name,
                "rows": len(split_df),
                "timestamp_min": split_df[data_config.timestamp_col].min(),
                "timestamp_max": split_df[data_config.timestamp_col].max(),
            }
        )

split_table = pd.DataFrame(split_rows)
split_table

In [ ]:
global_train_frame = pd.concat(
    [splits["train"] for splits in split_frames.values()],
    ignore_index=True,
)
scaler = fit_scaler_on_train(
    global_train_frame,
    feature_cols=data_config.feature_cols,
    scaler_type=SCALER_TYPE,
)

print(
    "global scaler | "
    f"fit_rows={len(global_train_frame)} | scaler_type={SCALER_TYPE} | "
    f"features={data_config.feature_cols}"
)

In [ ]:
scaled_frames = {}
scaled_rows = []

for ticker, splits in split_frames.items():
    scaled_frames[ticker] = {}
    for split_name, split_df in splits.items():
        scaled = transform_split(
            split_df,
            scaler=scaler,
            feature_cols=data_config.feature_cols,
        )
        scaled_frames[ticker][split_name] = scaled
        scaled_rows.append(
            {
                "ticker": ticker,
                "split": split_name,
                "rows": len(scaled),
                "feature_nulls": int(scaled[data_config.feature_cols].isna().sum().sum()),
            }
        )

scaled_table = pd.DataFrame(scaled_rows)
scaled_table

In [ ]:
ready_frames = {}
trim_rows = []

for ticker, splits in scaled_frames.items():
    ready_frames[ticker] = {}
    for split_name, split_df in splits.items():
        ready = trim_labels_at_split_boundary(
            split_df,
            label_horizon_k=window_config.label_horizon_k,
            label_col=LABEL_COL,
            timestamp_col=data_config.timestamp_col,
            ticker_col=TICKER_COL,
            timezone_policy=data_config.timezone_policy,
        )
        ready_frames[ticker][split_name] = ready
        trim_rows.append(
            {
                "ticker": ticker,
                "split": split_name,
                "rows": len(ready),
                "label_nan": int(ready[LABEL_COL].isna().sum()),
            }
        )

trim_table = pd.DataFrame(trim_rows)
trim_table

In [ ]:
datasets_by_ticker = {}
window_rows = []

for ticker, splits in ready_frames.items():
    datasets_by_ticker[ticker] = {}
    for split_name, split_df in splits.items():
        dataset = WindowedClassificationDataset(
            split_df,
            feature_cols=data_config.feature_cols,
            label_col=LABEL_COL,
            ticker_col=TICKER_COL,
            timestamp_col=data_config.timestamp_col,
            window_size=window_config.window_size,
            label_horizon_k=window_config.label_horizon_k,
            stride=window_config.stride,
        )
        datasets_by_ticker[ticker][split_name] = dataset
        window_rows.append(
            {
                "ticker": ticker,
                "split": split_name,
                "windows": len(dataset),
            }
        )

window_table = pd.DataFrame(window_rows)
if (window_table["windows"] == 0).any():
    zero_windows = window_table.loc[window_table["windows"] == 0, ["ticker", "split"]]
    raise ValueError(
        "one or more per-ticker split datasets have zero windows; adjust row cap, "
        f"window_size, or label_horizon_k: {zero_windows.to_dict(orient='records')}"
    )
window_table

In [ ]:
train_dataset = ConcatDataset([datasets_by_ticker[ticker]["train"] for ticker in TICKERS])
val_dataset = ConcatDataset([datasets_by_ticker[ticker]["val"] for ticker in TICKERS])
test_dataset = ConcatDataset([datasets_by_ticker[ticker]["test"] for ticker in TICKERS])

pooled_sizes = {
    "train": len(train_dataset),
    "val": len(val_dataset),
    "test": len(test_dataset),
}
print(
    "pooled datasets | "
    f"train={pooled_sizes['train']} | val={pooled_sizes['val']} | test={pooled_sizes['test']}"
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=train_config.batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=train_config.batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=train_config.batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

sample_x, sample_y = next(iter(train_loader))
print(
    "loaders | "
    f"batch_x_shape={tuple(sample_x.shape)} | batch_y_shape={tuple(sample_y.shape)} | "
    f"num_workers={NUM_WORKERS}"
)

In [ ]:
model = LSTMClassifier(**model_config.params)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=train_config.learning_rate,
    weight_decay=train_config.weight_decay,
)
criterion = torch.nn.CrossEntropyLoss()
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=None,
    device=train_config.device,
    checkpoint_dir=str(CHECKPOINT_ROOT / "main"),
    monitor_metric=train_config.monitor_metric,
    monitor_mode=train_config.monitor_mode,
    early_stop_patience=train_config.early_stop_patience,
    grad_clip=train_config.grad_clip,
    verbose=True,
)

param_count = sum(parameter.numel() for parameter in model.parameters())
print(
    "model/trainer | "
    f"model={model_config.name} | parameters={param_count} | "
    f"device={train_config.device}"
)

In [ ]:
history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=train_config.num_epochs,
)

print(
    "training complete | "
    f"best_epoch={history['best_epoch']} | "
    f"best_{train_config.monitor_metric}={history['best_metric']:.6f}"
)

In [ ]:
val_metrics, val_true, val_pred = evaluate(
    model,
    val_loader,
    criterion,
    train_config.device,
)
test_metrics, test_true, test_pred = evaluate(
    model,
    test_loader,
    criterion,
    train_config.device,
)

y_train = np.array([int(train_dataset[index][1].item()) for index in range(len(train_dataset))])
eval_labels_by_split = {"val": val_true, "test": test_true}

baseline_rows = []
for split_name, y_eval in eval_labels_by_split.items():
    stratified_runs = [
        dummy_baseline_metrics(
            y_train=y_train,
            y_test=y_eval,
            strategy="stratified",
            random_state=SEED + run_index,
        )
        for run_index in range(10)
    ]
    baseline_rows.append(
        {
            "model_name": "dummy_stratified",
            "split": split_name,
            "accuracy": float(np.mean([run["accuracy"] for run in stratified_runs])),
            "macro_f1": float(np.mean([run["macro_f1"] for run in stratified_runs])),
            "balanced_accuracy": float(np.mean([run["balanced_accuracy"] for run in stratified_runs])),
            "macro_f1_std": float(np.std([run["macro_f1"] for run in stratified_runs], ddof=0)),
            "notes": "mean over 10 seeds",
        }
    )

    prior_metrics = dummy_baseline_metrics(
        y_train=y_train,
        y_test=y_eval,
        strategy="prior",
        random_state=SEED,
    )
    baseline_rows.append(
        {
            "model_name": "dummy_prior",
            "split": split_name,
            "accuracy": prior_metrics["accuracy"],
            "macro_f1": prior_metrics["macro_f1"],
            "balanced_accuracy": prior_metrics["balanced_accuracy"],
            "macro_f1_std": np.nan,
            "notes": "fit on pooled train labels",
        }
    )

    for model_name, constant_label in (("always_up", 1), ("always_down", 0)):
        constant_metrics = always_predict_baseline_metrics(
            y_test=y_eval,
            constant_label=constant_label,
        )
        baseline_rows.append(
            {
                "model_name": model_name,
                "split": split_name,
                "accuracy": constant_metrics["accuracy"],
                "macro_f1": constant_metrics["macro_f1"],
                "balanced_accuracy": constant_metrics["balanced_accuracy"],
                "macro_f1_std": np.nan,
                "notes": f"constant_label={constant_label}",
            }
        )

model_rows = pd.DataFrame(
    [
        {
            "model_name": "lstm_classifier",
            "split": "val",
            "accuracy": val_metrics["accuracy"],
            "macro_f1": val_metrics["macro_f1"],
            "balanced_accuracy": val_metrics["balanced_accuracy"],
            "macro_f1_std": np.nan,
            "notes": "pooled smoke",
        },
        {
            "model_name": "lstm_classifier",
            "split": "test",
            "accuracy": test_metrics["accuracy"],
            "macro_f1": test_metrics["macro_f1"],
            "balanced_accuracy": test_metrics["balanced_accuracy"],
            "macro_f1_std": np.nan,
            "notes": "pooled smoke",
        },
    ]
)
baseline_table = pd.DataFrame(baseline_rows)
comparison_table = pd.concat([model_rows, baseline_table], ignore_index=True)
comparison_table["delta_macro_f1_vs_dummy_stratified"] = np.nan
for split_name in comparison_table["split"].unique():
    dummy_macro_f1 = comparison_table.loc[
        (comparison_table["split"] == split_name)
        & (comparison_table["model_name"] == "dummy_stratified"),
        "macro_f1",
    ].iloc[0]
    split_mask = comparison_table["split"] == split_name
    comparison_table.loc[split_mask, "delta_macro_f1_vs_dummy_stratified"] = (
        comparison_table.loc[split_mask, "macro_f1"] - dummy_macro_f1
    )

comparison_table = comparison_table[
    [
        "model_name",
        "split",
        "accuracy",
        "macro_f1",
        "balanced_accuracy",
        "delta_macro_f1_vs_dummy_stratified",
        "notes",
    ]
]
comparison_table

In [ ]:
seed_everything(SEED, deterministic=True)
train_x = torch.stack([train_dataset[index][0] for index in range(len(train_dataset))])
train_y = torch.stack([train_dataset[index][1] for index in range(len(train_dataset))])
shuffle_generator = torch.Generator().manual_seed(SEED)
shuffle_order = torch.randperm(len(train_y), generator=shuffle_generator)
shuffled_train_dataset = TensorDataset(train_x, train_y[shuffle_order])
shuffled_train_loader = DataLoader(
    shuffled_train_dataset,
    batch_size=train_config.batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

shuffled_model = LSTMClassifier(**model_config.params)
shuffled_optimizer = torch.optim.Adam(
    shuffled_model.parameters(),
    lr=train_config.learning_rate,
    weight_decay=train_config.weight_decay,
)
shuffled_trainer = Trainer(
    model=shuffled_model,
    optimizer=shuffled_optimizer,
    criterion=criterion,
    scheduler=None,
    device=train_config.device,
    checkpoint_dir=str(CHECKPOINT_ROOT / "shuffled_labels"),
    monitor_metric=train_config.monitor_metric,
    monitor_mode=train_config.monitor_mode,
    early_stop_patience=train_config.early_stop_patience,
    grad_clip=train_config.grad_clip,
    verbose=True,
)
shuffled_history = shuffled_trainer.fit(
    train_loader=shuffled_train_loader,
    val_loader=val_loader,
    num_epochs=train_config.num_epochs,
)
shuffled_val_metrics, shuffled_val_true, shuffled_val_pred = evaluate(
    shuffled_model,
    val_loader,
    criterion,
    train_config.device,
)

dummy_stratified_macro_f1_mean = comparison_table.loc[
    (comparison_table["split"] == "val")
    & (comparison_table["model_name"] == "dummy_stratified"),
    "macro_f1",
].iloc[0]
shuffled_label_macro_f1 = shuffled_val_metrics["macro_f1"]
if shuffled_label_macro_f1 <= 1.10 * dummy_stratified_macro_f1_mean:
    sanity_status = "PASS"
elif shuffled_label_macro_f1 <= 1.20 * dummy_stratified_macro_f1_mean:
    sanity_status = "WARNING"
else:
    sanity_status = "FAIL"

sanity_table = pd.DataFrame(
    [
        {
            "check": "shuffled_label_sanity",
            "status": sanity_status,
            "shuffled_label_macro_f1": shuffled_label_macro_f1,
            "dummy_stratified_macro_f1_mean": dummy_stratified_macro_f1_mean,
            "pass_threshold": 1.10 * dummy_stratified_macro_f1_mean,
            "warning_threshold": 1.20 * dummy_stratified_macro_f1_mean,
            "best_epoch": shuffled_history["best_epoch"],
        }
    ]
)
sanity_table

## Interpretation notes

- This notebook is a smoke test for pooled orchestration, not a final benchmark.
- Windows are built per ticker before pooling with `ConcatDataset`.
- The scaler is fit once on concatenated train splits only, then reused for each ticker and split.
- Public API blockers should be fixed in `ml_utils` with tests before changing this notebook.